In [2]:
!pip install sinabs


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np

folder_path = r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\saved_data\ROAD"

int_id_data = np.load(folder_path + r"\int_id_data.npy", allow_pickle=True)
attack_labels = np.load(folder_path + r"\attack_labels.npy", allow_pickle=True)


int_id_data_test = np.load(folder_path + r"\int_id_data_test.npy", allow_pickle=True)
attack_labels_test = np.load(folder_path + r"\attack_labels_test.npy", allow_pickle=True)

In [4]:
import builtins
import numpy as np


def sliding_windows_id_data(samples, labels, window_size=32, step=2, attack_labels=('T', 'D', 'F', 'S')):
    n = len(samples)

    if isinstance(attack_labels, str):
        attack_labels = (attack_labels,)

    windows = []
    window_labels = []
    next_id_datas = []
    

    for start in range(0, n - window_size, step):
        end = start + window_size
        window_slice = labels[start:end]

        windows.append(samples[start:end])
        next_id_datas.append(samples[end])

        matching_attack = builtins.next(
            (label for label in attack_labels if label in window_slice),
            'R'
        )
        window_labels.append(matching_attack)

    return (
        np.array(windows, dtype=np.uint8),
        np.array(next_id_datas, dtype=object),
        np.array(window_labels, dtype=object)
    )

In [5]:
windows, windowed_labels = [], []
for i in range(len(int_id_data)):
    windows_batch, next_values, windowed_labels_batch = sliding_windows_id_data(int_id_data[i], attack_labels[i])
    windows.extend(windows_batch)
    windowed_labels.extend(windowed_labels_batch)


In [6]:
windowed_labels = np.array(windowed_labels)
attack_onehot = np.zeros((len(windowed_labels)), dtype=np.uint8)
attack_onehot[windowed_labels == 'T'] = 1

print(attack_onehot[:10])

[0 0 0 0 0 0 0 0 0 0]


In [7]:
windows_test, windowed_labels_test = [], []
for i in range(len(int_id_data)):
    windows_batch, next_values, windowed_labels_batch = sliding_windows_id_data(int_id_data_test[i], attack_labels_test[i])
    windows_test.extend(windows_batch)
    windowed_labels_test.extend(windowed_labels_batch)

In [8]:
windowed_labels_test = np.array(windowed_labels_test)

attack_onehot_test = np.zeros((len(windowed_labels_test)), dtype=np.uint8)
attack_onehot_test[windowed_labels_test == 'T'] = 1

print(attack_onehot_test[:10])

[0 0 0 0 0 0 0 0 0 0]


In [9]:
import torch
from torch.utils.data import Dataset


class CANDataset(Dataset):
    def __init__(
        self,
        windows,
        attack_onehot,
        train=True,
        is_spiking=False,
        time_window=100,
    ):
        # Convert to tensors
        self.windows = torch.as_tensor(windows, dtype=torch.float32)

        # Add channel dimension:
        # (N, H, W) -> (N, 1, H, W)
        if self.windows.ndim == 3:
            self.windows = self.windows.unsqueeze(1)

        self.labels = torch.as_tensor(attack_onehot, dtype=torch.long)

        self.is_spiking = is_spiking
        self.time_window = time_window

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):

        img = self.windows[index]
        label = self.labels[index]

        if self.is_spiking:
            # Optional normalization if inputs are not already in [0,1]
            #img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            # Generate spike train
            img = (
                torch.rand(self.time_window, *img.shape) < img
            ).float()

        return img, label

In [10]:
windows = np.array(windows)
print(windows.shape)
print(windows.dtype)
print(windows.min(), windows.max())

print(windows[0])

windows_test = np.array(windows_test)
print(windows_test.shape)
print(windows_test.dtype)
print(windows_test.min(), windows_test.max())

print(windows_test[0])

(2890134, 32, 10)
uint8
0 255
[[  1  83   0   0   0   0   0  12   0  10]
 [  5 225 137  47  96  11  10   1   0 128]
 [  6  98  78 224   0   0  64   0   0   0]
 [  1 156   0 126  32  16   2   0  40 240]
 [  0 208   2 119   4  96  95   3 154   0]
 [  6 158   4  64   4 125  31 192  21  98]
 [  1  37 144   0  65  31  64  63 109  96]
 [  2 193   1 244  15 199 203  31  82 154]
 [  0  51   0   6 120   0  12   2 215 208]
 [  2 116   0 242  32 252 144 124 141 217]
 [  0 192  96   0   0   0   0   0   0   0]
 [  1  98   0   8   4  19 234  17 244 206]
 [  0 167   0  16 249 164 193  46  16 160]
 [  0  14  32  82  86   2   8   9 118 214]
 [  3 193 123  85  31 186 210  83 105  46]
 [  1   7   0   0   0   0   0   0   0   0]
 [ 15 255   0   0   0   0   0   0   0   0]
 [  6 224   3  31   3  27   3  34   3  33]
 [  3  84  32  19  64   0   0   1 136 128]
 [  5 225 137  47  96  11  10   1   0 128]
 [  2 149   0   0   0   0   0   0  24  64]
 [  2 139   0   0   0   0   0   0   0   0]
 [  0 208  10 119   4  9

In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

windows = scaler.fit_transform(
    windows.reshape(-1, windows.shape[-1])
).reshape(windows.shape)

windows_test = scaler.transform(
    windows_test.reshape(-1, windows_test.shape[-1])
).reshape(windows_test.shape)

In [12]:
from torch.utils.data import DataLoader

canbus_train = CANDataset(windows, attack_onehot, train=True, is_spiking=False)
train_loader = DataLoader(canbus_train, batch_size=256, shuffle=True)

canbus_test = CANDataset(windows_test, attack_onehot_test, train=False, is_spiking=False)
test_loader = DataLoader(canbus_test, batch_size=256, shuffle=False)

In [14]:
test_batch_size = 10
num_timesteps = 100

spike_mnist_test = CANDataset(
    windows_test, attack_onehot_test, train=False, is_spiking=True, time_window=num_timesteps
)
spike_test_loader = DataLoader(
    spike_mnist_test, batch_size=test_batch_size, shuffle=True
)

In [15]:
print(spike_test_loader)

batch = next(iter(spike_test_loader))
inputs, labels = batch

print("inputs type:", type(inputs))
print("labels type:", type(labels))
print("inputs shape:", inputs.shape)
print("labels shape:", labels.shape)
print("inputs dtype:", inputs.dtype)
print("labels dtype:", labels.dtype)

print("first input sample:")
print(inputs[0])

print("first label batch:")
print(labels[:10])

inputs type: <class 'torch.Tensor'>
labels type: <class 'torch.Tensor'>
inputs shape: torch.Size([10, 100, 1, 32, 10])
labels shape: torch.Size([10])
inputs dtype: torch.float32
labels dtype: torch.int64
first input sample:
tensor([[[[0., 0., 0.,  ..., 0., 1., 0.],
          [1., 1., 0.,  ..., 0., 0., 0.],
          [1., 1., 0.,  ..., 1., 0., 1.],
          ...,
          [0., 1., 0.,  ..., 0., 0., 1.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [1., 1., 1.,  ..., 0., 0., 1.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [1., 1., 0.,  ..., 0., 0., 0.],
          [1., 1., 0.,  ..., 0., 0., 1.],
          ...,
          [0., 1., 0.,  ..., 0., 0., 1.],
          [1., 0., 0.,  ..., 0., 0., 0.],
          [1., 1., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 1., 1., 1.],
          [1., 1., 0.,  ..., 0., 0., 0.],
          [0., 1., 0.,  ..., 1., 0., 0.],
          ...,
          [1., 1., 0.,  ..., 0., 0., 0.],
          [0., 1., 1.,  ..., 0., 1., 1.],
         

In [ ]:
import torch

# Save the trained ANN weights
torch.save(ann.state_dict(), "ann_weights.pth")

In [ ]:
# Build your BrainScaleS model first
brainscales_model = build_brainscales_model()

# Load the saved ANN weights
ann_weights = torch.load("ann_weights.pth", map_location="cpu")

In [ ]:
brainscales_model[0].weight.data.copy_(ann_weights["0.weight"])
brainscales_model[0].bias.data.copy_(ann_weights["0.bias"])

brainscales_model[3].weight.data.copy_(ann_weights["3.weight"])
brainscales_model[3].bias.data.copy_(ann_weights["3.bias"])

brainscales_model[6].weight.data.copy_(ann_weights["6.weight"])
brainscales_model[6].bias.data.copy_(ann_weights["6.bias"])

brainscales_model[8].weight.data.copy_(ann_weights["8.weight"])
brainscales_model[8].bias.data.copy_(ann_weights["8.bias"])

In [ ]:
def rate_encode(x, T):

    rand = torch.rand((T,) + x.shape)

    spikes = rand < x

    return spikes.float()

In [ ]:
state = None

for t in range(T):

    spikes = data[t]

    output, state = brainscales_model(spikes, state)